# Customer Support Intent Evaluation Using LangSmith
Final v3 and judge rubric v2. Execute all cells offline from the project root or v3 directory. No API calls. Historical scores and human reviews remain separate.

## Sources and completed review
Sources: Evaluations Using LangSmith handout and The Gen Academy repository. 100 synthetic tickets from 25 seed scenarios, each original plus three paraphrases. Five intent categories. Historical Excel reviews: Evaluation 11/11 and Fresh evaluation 14/14 approved. Completed LangSmith queues: baseline 8/8 and focused 7/7, 15 run reviews across 14 tickets with 75 feedback records. Approvals document review; they do not rewrite predictions.

In [1]:
from pathlib import Path
import json,hashlib
ROOT=Path.cwd()
if not (ROOT/'results/comparison.json').exists():
    ROOT=next(p for p in [ROOT/'v3',ROOT.parent/'v3',ROOT/'p4/v3'] if (p/'results/comparison.json').exists())
summary=json.loads((ROOT/'results/comparison.json').read_text(encoding='utf-8'))
rows=json.loads((ROOT/'results/ticket_comparison.json').read_text(encoding='utf-8'))
assert len(rows)==100 and len({r['id'] for r in rows})==100
print('Completed:',summary['completed'],'Errors:',summary['errors'])
print('Requested model:',summary['model_requested'],'Returned models:',summary['returned_models'])

Completed: 100 Errors: 0
Requested model: gpt-4o-mini Returned models: ['gpt-4o-mini-2024-07-18']


## Separate the label correction from model improvement
Historical reviewed labels gave93/100 versus93/100. User-approved damaged-product routing changes t046 to product_issue. Every version below is scored against that same new policy. Baseline and focused are saved historical predictions; only v3 makes new model calls.

In [2]:
for version,pred in [('baseline','baseline_prediction'),('focused','focused_prediction'),('v3','v3_prediction')]:
    correct=sum(r[pred]==r['v3_policy_intent'] for r in rows)
    assert correct==summary['same_v3_policy_scores'][version]
    print(version,correct,'/100')
print('Reference-policy effect before new calls:',summary['label_policy_effect_before_new_calls'])
print('V3 vs focused:',summary['v3_paired_changes'])
assert summary['v3_paired_changes']=={'FIXED':6,'REGRESSION':0,'STILL_PASS':94,'STILL_FAIL':0}

baseline 92 /100
focused 94 /100
v3 100 /100
Reference-policy effect before new calls: {'baseline': -1, 'focused': 1}
V3 vs focused: {'FIXED': 6, 'REGRESSION': 0, 'STILL_PASS': 94, 'STILL_FAIL': 0}


## Exactly what changed
Six outputs changed from order_status to refund_request. The t046 model prediction stayed product_issue; only its new reference policy changed. Original workbook notes, approvals and historical labels are preserved.

In [3]:
for r in rows:
    if r['label_changed'] or r['prediction_changed']:
        print(r['id'],'| earlier reference:',r['historical_reviewed_intent'],'| v3 policy:',r['v3_policy_intent'],'| focused:',r['focused_prediction'],'| v3:',r['v3_prediction'])
assert [r['id'] for r in rows if r['prediction_changed']]==['t024','t025','t026','t027','t028','t029']
assert [r['id'] for r in rows if r['label_changed']]==['t046']

t024 | earlier reference: refund_request | v3 policy: refund_request | focused: order_status | v3: refund_request
t025 | earlier reference: refund_request | v3 policy: refund_request | focused: order_status | v3: refund_request
t026 | earlier reference: refund_request | v3 policy: refund_request | focused: order_status | v3: refund_request
t027 | earlier reference: refund_request | v3 policy: refund_request | focused: order_status | v3: refund_request
t028 | earlier reference: refund_request | v3 policy: refund_request | focused: order_status | v3: refund_request
t029 | earlier reference: refund_request | v3 policy: refund_request | focused: order_status | v3: refund_request
t046 | earlier reference: refund_request | v3 policy: product_issue | focused: product_issue | v3: product_issue


## Preservation and limits
100% applies only to the known100 synthetic tickets, derived from25 seed families. No independent test is included. A separate LLM judge run is documented below when present. The cloud-recording status is separate from local model execution. Human review has not automatically approved the new v3 outputs.

In [4]:
freeze=json.loads((ROOT/'inputs/freeze.json').read_text(encoding='utf-8'))
source=ROOT.parent/'customer-support'
if not source.exists(): source=ROOT.parent
for name,h in freeze['source_hashes'].items():
    assert hashlib.sha256((source/name).read_bytes()).hexdigest()==h,name
print('Historical workbook, predictions, prompts and annotation evidence unchanged.')
for r in rows:
    raw=json.loads((ROOT/'runs'/(r['id']+'.json')).read_text(encoding='utf-8'))
    assert raw['prediction']==r['v3_prediction']
    assert raw['request']['messages'][1]['content']==r['query']
    assert raw['request']['model']=='gpt-4o-mini' and raw['request']['temperature']==0
print('All100 saved predictions match actual response records; labels are not sent as query data.')
print(json.loads((ROOT/'results/langsmith_sync_status.json').read_text()))

Historical workbook, predictions, prompts and annotation evidence unchanged.
All100 saved predictions match actual response records; labels are not sent as query data.
{'status': 'VERIFIED', 'reason': None, 'new_classifier_calls_for_sync': 0, 'judge_feedback_verified': True, 'calibrated_judge_and_human_review_verified': True}


## Completed LLM judge:100 Exact Match passes versus99 judge passes
The same frozen prior rubric and settings evaluate only query plus predicted category. Nine ambiguity flags include the one disagreement,t033. Its overcharge/refund wording causes the judge to choose account_help, while the reference and classifier choose refund_request. Prior identical-input judge variability is retained as a reliability limitation. Do not automatically change the label to force agreement.

In [5]:
judge=json.loads((ROOT/'llm_judge/summary.json').read_text(encoding='utf-8'))
assessments=json.loads((ROOT/'llm_judge/assessments.json').read_text(encoding='utf-8'))
assert judge['valid_assessments']==100 and sum(r['score'] for r in assessments)==99
assert judge['score_disagreement_ids']==['t033']
assert judge['prior_judgments_and_v3_predictions_unchanged'] and judge['rubric_identical_to_original']
print('Historical judge:',judge['historical_judge_pass'],'V3 judge:',judge['judge_pass'],'/100')
print('Ambiguity flags:',judge['ambiguity_flags'],'Review tickets:',judge['review_ticket_ids'])
print('t033:',next(r for r in assessments if r['id']=='t033'))
print('Cloud feedback verified:',json.loads((ROOT/'results/langsmith_v3_evidence.json').read_text())['judge'])

Historical judge: {'baseline': 92, 'focused': 93} V3 judge: 99 /100
Ambiguity flags: 9 Review tickets: ['t004', 't005', 't006', 't007', 't033', 't044', 't045', 't046', 't047']
t033: {'id': 't033', 'query': "I can't believe I was charged $89 when the checkout clearly said $79. This is ridiculous! Please refund the extra $10 immediately!", 'prediction': 'refund_request', 'score': 0, 'correctness': 'incorrect', 'rationale': 'The complaint is about a checkout/payment discrepancy where the customer was charged more than the stated price, which fits account_help as an actual checkout failure, even though a refund is requested as the remedy.', 'ambiguity': True, 'proposed_intent': 'account_help', 'alternative_intents': 'refund_request', 'approved_policy_reference': 'refund_request', 'exact_match': 1, 'judge_agrees_with_exact_match': False, 'judge_agrees_with_reference': False, 'needs_review': True}
Cloud feedback verified: {'feedback_key': 'week4_p4_v3_llm_judge', 'verified_records': 100, 'co

## Judge rubric v2 and separately saved human adjudication
The participant accepted refund_request for t033. A general billing/refund clarification was added to the rubric, then all100 saved outputs were assessed again. Classifier outputs, schema, model and settings remain unchanged. Old scores are not edited. This calibrates a known disagreement; it is not independent held-out validation. Twelve ambiguity flags can coexist with100 rubric passes.

In [6]:
calibration=json.loads((ROOT/'llm_judge_v2/before_after.json').read_text(encoding='utf-8'))
new=json.loads((ROOT/'llm_judge_v2/assessments.json').read_text(encoding='utf-8'))
human=json.loads((ROOT/'human_reviews/t033.json').read_text(encoding='utf-8'))
assert len(new)==100 and sum(r['score'] for r in new)==100
assert calibration['before_judge_pass']==99 and calibration['after_judge_pass']==100
assert calibration['paired_changes']=={'WIN':1,'REGRESSION':0,'UNCHANGED_PASS':99,'UNCHANGED_FAIL':0}
assert human['approved_intent']=='refund_request' and human['human_adjudicated_score']==1
old33=json.loads((ROOT/'llm_judge/judgments/t033.json').read_text(encoding='utf-8'))
assert old33['judgment']['score']==0 and hashlib.sha256((ROOT/'llm_judge/judgments/t033.json').read_bytes()).hexdigest()==human['old_judgment_sha256']
print('Before:',calibration['before_judge_pass'],'After:',calibration['after_judge_pass'],'Classifier:',100,'→',100)
print('Changed score IDs:',calibration['score_changed_ids'],'Remaining ambiguity flags:',calibration['new_ambiguity_flags'])
print('Separate user review:',human['note'])
print('Cloud rubric-v2 and human review:',json.loads((ROOT/'results/langsmith_v3_evidence.json').read_text())['judge_v2'])

Before: 99 After: 100 Classifier: 100 → 100
Changed score IDs: ['t033'] Remaining ambiguity flags: 12
Separate user review: Human review: refund_request is correct. The customer disputes an overcharge and requests reimbursement. Mentioning checkout does not establish a technical checkout failure. The v3 prediction is accepted.
Cloud rubric-v2 and human review: {'feedback_key': 'week4_p4_v3_llm_judge_v2', 'verified': True, 'verified_records': 100, 'correct': 100, 'ambiguity_flags': 12, 'disagreement_ids': [], 'before_judge_correct': 99, 'delta_percentage_points': 1}
